# given a fixed model, how "good" is it?


### 评估语言模型的重要性与框架

你可能认为评估是一个机械化的过程（拿现有的模型，给它丢一些提示，计算一些平均值...）
其实，评估是一个深刻且丰富的话题...
它决定了语言模型的未来。

#### 评估的意义是什么？

没有一个统一的评估标准，它取决于你试图回答的具体问题。

1. **用户或公司**希望基于其用例（例如客服聊天机器人）做出购买决策（模型 A 还是模型 B）。
2. **研究人员**希望衡量模型的原始能力（例如，智能程度）。
3. 我们想要了解模型的**好处与危害**（对业务和政策的影响）。
4. **模型开发者**希望获得反馈以改进模型。

在每种情况下，都有一个抽象的**目标**，需要转化为具体的评估方式。

---

### 评估框架

1. **输入是什么？**
2. 如何**调用**语言模型？
3. 如何评估**输出**？
4. 如何**解读**结果？

---

#### 1. 输入是什么？

* 1.1 **覆盖的使用场景**有哪些？
* 1.2 输入中是否包括**困难的输入**（例如，尾部数据）？
* 1.3 输入是否**适配**于模型（例如，多轮对话）？

#### 2. 如何调用语言模型？

* 2.1 如何为语言模型编写提示（prompt）？
* 2.2 语言模型是否使用了链式思维（chain-of-thought）、工具、RAG 等？
* 2.3 我们是在评估语言模型还是一个智能系统（模型开发者更关心前者，用户更关心后者）？

#### 3. 如何评估输出？

* 3.1 用于评估的参考输出是否**无误**？
* 3.2 使用哪些评估指标（例如，pass\@k）？
* 3.3 如何考虑**成本**（例如，推理与训练的成本）？
* 3.4 如何考虑**不对称的错误**（例如，在医疗场景中的幻觉问题）？
* 3.5 如何处理**开放式生成**（没有明确的标准答案）？

#### 4. 如何解读评估结果？

* 4.1 如何解读一个数字（例如 91%）— 它是否准备好部署？
* 4.2 如何评估面对训练与测试数据重叠时的**泛化能力**？
* 4.3 我们是在评估**最终模型**还是评估**方法**？


### 语言模型与困惑度（Perplexity）



困惑度是衡量一个概率模型预测样本好坏程度的指标，在自然语言处理（NLP）中，它特指语言模型（Language Model, LM）在预测一个词序列时的不确定性或“惊讶”程度。

**核心思想：一个好的语言模型应该对它要预测的文本（测试集）感到不那么“惊讶”或“困惑”。困惑度越低，说明模型对文本的概率分布建模得越好，性能也就越优。**

#### 观理解 (Intuitive Understanding)

你可以将困惑度理解为模型在预测下一个词时，平均有多少个“合理”的选项。

  * **PPL = 10**：意味着模型在预测下一个词时，其不确定性等价于从 10 个等概率的词中进行选择。
  * **PPL = 1**：这是一个理想化的完美模型，对于每个词的预测都百分之百确定，没有任何困惑。
  * **PPL = |V|** (词汇表大小)：这相当于一个最差的模型，它对所有词都给出了相同的均匀概率（即随机猜测），其困惑度等于整个词汇表的大小。

因此，我们的目标是训练一个模型，使其在未见过的测试集上获得尽可能低的困惑度。


#### 标准数据集

* Penn Treebank (WSJ)
* WikiText-103 (Wikipedia)
* One Billion Word Benchmark (来自机器翻译 WMT11 - EuroParl, UN, news)

有些论文在同一个数据集（训练集和测试集）上进行训练和评估。

#### 示例：

* **纯 CNNs+LSTMs 在 One Billion Word Benchmark 上的困惑度**：从 51.3 降到 30.0  ([论文链接](https://arxiv.org/abs/1602.02410))
* **GPT-2**：在 WebText 上训练（40GB 文本，来自 Reddit 链接的网页），在标准数据集上进行零样本评估。

这属于**超出分布的评估**（out-of-distribution evaluation），但这种方法的基本思想是，训练数据已经涵盖了大量信息。


#### 困惑度的表现

* 在小数据集上效果更好（迁移学习有帮助），但在较大的数据集（例如 1BW）上效果较差。
* 自 GPT-2 和 GPT-3 以来，语言建模的研究逐渐更多地关注下游任务的准确率。

#### 困惑度依然有用的原因：

* 比下游任务的准确率更平滑（适合拟合规模定律）。
* 是通用的（这也是我们用它进行训练的原因），而任务准确率可能会遗漏一些细微的差别。
* **注意**：也可以在下游任务上测量条件困惑度（用于规模定律的研究） ([论文链接](https://arxiv.org/abs/2412.04403))。

#### 评估困惑度的警告

如果你在运行排行榜，评估者需要信任语言模型。

* 对于任务准确率，可以直接取出黑箱模型的输出并计算所需的指标。
* 对于困惑度，语言模型需要生成概率，并信任这些概率的和为 1（尤其是早期的 UNKs 问题更加复杂）。

#### 困惑度的极端观点

* 你的真实分布是 $t$，模型是 $p$。
* 最佳的困惑度是 $H(t)$，当且仅当 $p = t$ 时获得。
* 如果我们知道 $t$，就能解决所有任务。
* 所以，通过降低困惑度，最终将达到 AGI（人工通用智能）。
* **警告**：这可能不是最有效的方式来实现目标（可能会降低那些不重要部分的困惑度）。

#### 与困惑度相关的任务

类似的概念：**cloze** 任务，例如 **LAMBADA** ([论文链接](https://arxiv.org/abs/1606.06031))。



另一个例子：**HellaSwag** ([论文链接](https://arxiv.org/pdf/1905.07830))。


